In [ ]:
# ── Cell 0: clone/update thesis repo from GitHub ────────────────────────────
import subprocess, os

REPO_URL = "https://github.com/richardtekere09/brain-tumor-mri-thesis.git"

if os.path.exists(".git"):
    # Already a git repo — just pull latest
    subprocess.run(["git", "pull"], check=True)
    print("Repo updated via git pull.")
elif os.path.exists("train.py"):
    # Files present but not a git repo — init and pull
    subprocess.run(["git", "init"], check=True)
    subprocess.run(["git", "remote", "add", "origin", REPO_URL], check=True)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", "-f", "origin/main"], check=True)
    print("Repo initialised and checked out.")
else:
    # Empty or near-empty directory — clone into it
    subprocess.run(["git", "clone", REPO_URL, "repo_tmp"], check=True)
    import shutil
    for item in os.listdir("repo_tmp"):
        shutil.move(os.path.join("repo_tmp", item), item)
    shutil.rmtree("repo_tmp")
    print("Repo cloned.")


# Brain Tumor MRI — Training Notebook
**Thesis:** Нейросетевой анализ МРТ-изображений головного мозга для диагностики заболеваний мозга

Run this notebook on Kaggle GPU (P100/T4).  
Set `MODEL` to `model_a`, `model_b`, or `model_c` before running all cells.

In [ ]:
# ── Cell 1: choose model ─────────────────────────────────────────────────────
MODEL = 'model_a'   # change to 'model_b' or 'model_c'
print(f'Training: {MODEL}')

In [ ]:
# ── Cell 2: install dependencies ─────────────────────────────────────────────
import subprocess, sys

packages = [
    'monai>=1.3',
    'transformers>=4.35',
    'nibabel>=5.0',
    'einops',
    'timm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('Dependencies installed.')

In [ ]:
# ── Cell 3: mount data and set up paths ──────────────────────────────────────
import os, shutil, pathlib

# ── Kaggle dataset paths ───────────────────────────────────────────────────
# Public BraTS dataset (awsaf49)
BRATS_INPUT     = '/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation'
# Your uploaded datasets (isaactekere)
USER_BASE       = '/kaggle/input/datasets/isaactekere'
TEXTBRATS_INPUT = f'{USER_BASE}/textbrats/textbrats/TextBraTSData'
SPLITS_INPUT    = f'{USER_BASE}/brain-tumor-splits/brain-tumor-splits'
WEIGHTS_INPUT   = f'{USER_BASE}/brain-tumor-weights/brain-tumor-weights'
# ──────────────────────────────────────────────────────────────────────────

os.makedirs('data', exist_ok=True)
os.makedirs('pretrained', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# Symlink BraTS data
brats_link = 'data/BraTS2020_TrainingData'
if not os.path.lexists(brats_link):
    # Try nested MICCAI folder first, then direct
    for candidate in [
        os.path.join(BRATS_INPUT, 'BraTS2020_TrainingData', 'MICCAI_BraTS2020_TrainingData'),
        os.path.join(BRATS_INPUT, 'MICCAI_BraTS2020_TrainingData'),
        os.path.join(BRATS_INPUT, 'BraTS2020_TrainingData'),
        BRATS_INPUT,
    ]:
        if os.path.exists(candidate):
            os.symlink(candidate, brats_link)
            print(f'Symlinked BraTS: {candidate}')
            break
    else:
        raise FileNotFoundError(f'BraTS data not found under {BRATS_INPUT}')
else:
    print(f'BraTS symlink already exists: {brats_link}')

# Symlink TextBraTS features
text_link = 'data/TextBraTS_hf'
if not os.path.lexists(text_link):
    # Try with and without TextBraTSData subfolder
    for candidate in [TEXTBRATS_INPUT, os.path.dirname(TEXTBRATS_INPUT)]:
        if os.path.exists(candidate):
            os.symlink(candidate, text_link)
            print(f'Symlinked TextBraTS: {candidate}')
            break
    else:
        raise FileNotFoundError(f'TextBraTS data not found at {TEXTBRATS_INPUT}')
else:
    print(f'TextBraTS symlink already exists: {text_link}')

# Copy splits.json
for splits_candidate in [
    os.path.join(SPLITS_INPUT, 'splits.json'),
    os.path.join(USER_BASE, 'brain-tumor-splits', 'splits.json'),
]:
    if os.path.exists(splits_candidate):
        shutil.copy2(splits_candidate, 'data/splits.json')
        print(f'Copied splits.json from {splits_candidate}')
        break
else:
    raise FileNotFoundError(f'splits.json not found. Checked: {SPLITS_INPUT}')

# Copy pretrained weights
for wf in ['resnet_18_23dataset.pth', 'swin_unetr_brats.pt']:
    dst = os.path.join('pretrained', wf)
    for weights_candidate in [
        os.path.join(WEIGHTS_INPUT, wf),
        os.path.join(USER_BASE, 'brain-tumor-weights', wf),
    ]:
        if os.path.exists(weights_candidate) and not os.path.exists(dst):
            shutil.copy2(weights_candidate, dst)
            print(f'Copied {wf}')
            break

print('\nData setup complete.')
print(f'  BraTS cases: {len(os.listdir(brats_link)) if os.path.exists(brats_link) else "MISSING"}')

In [ ]:
# ── Cell 4: verify data accessible ───────────────────────────────────────────
import os, json

cases = sorted(os.listdir('data/BraTS2020_TrainingData'))
print(f'BraTS cases found: {len(cases)}')

with open('data/splits.json') as f:
    splits = json.load(f)
print(f'Splits — train: {len(splits["train"])} | val: {len(splits["val"])} | test: {len(splits["test"])}')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# ── Cell 5: resume-aware training ────────────────────────────────────────────
import subprocess, sys

cmd = [
    sys.executable, 'train.py',
    '--config', f'configs/{MODEL}.yaml',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=False)
print(f'Exit code: {result.returncode}')

In [ ]:
# ── Cell 6: copy checkpoints to /kaggle/working/ ────────────────────────────
import shutil, os

src_dir = f'checkpoints/{MODEL}'
dst_dir = f'/kaggle/working/checkpoints/{MODEL}'
os.makedirs(dst_dir, exist_ok=True)

for fname in os.listdir(src_dir):
    shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))
    print(f'Copied: {fname}')

print(f'Checkpoints available for download in: {dst_dir}')

In [ ]:
# ── Cell 7: plot training curves ─────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

log_path = f'checkpoints/{MODEL}/train_log.csv'
if os.path.exists(log_path):
    df = pd.read_csv(log_path)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(df['epoch'], df['train_loss'], label='Train loss', color='steelblue')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'{MODEL} — Training Loss'); ax1.legend()

    for col, label in [('val_wt','WT'), ('val_tc','TC'), ('val_et','ET'), ('val_mean','Mean')]:
        ax2.plot(df['epoch'], df[col], label=label)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice')
    ax2.set_title(f'{MODEL} — Validation Dice'); ax2.legend()

    plt.tight_layout()
    out_png = f'/kaggle/working/training_curves_{MODEL}.png'
    plt.savefig(out_png, dpi=120)
    plt.show()
    print(f'Saved: {out_png}')
else:
    print('No train_log.csv found yet.')